# Quantium Virtual Internship — Task 1
## Retail Strategy and Analytics: Análisis exploratorio de datos (EDA) en Python

Este notebook es la traducción a Python del template en R Markdown (`InsideSherpa_Task1_DraftSolutions`).
Cada línea de código incluye un comentario explicando qué hace y por qué, para que puedas
seguir el razonamiento del análisis paso a paso.

**Datasets requeridos** (colócalos en la misma carpeta que este notebook):
- `QVI_transaction_data.csv`
- `QVI_purchase_behaviour.csv`


## 0. Cargar librerías

In [ ]:
# pandas: manipulación y análisis de datos tabulares (equivalente a data.table en R)
import pandas as pd

# numpy: operaciones numéricas de soporte (usado internamente por pandas y para cálculos)
import numpy as np

# re: expresiones regulares, para limpiar texto (nombres de producto)
import re

# Counter: para contar frecuencias de palabras de forma sencilla
from collections import Counter

# matplotlib: motor base de gráficos
import matplotlib.pyplot as plt

# seaborn: gráficos estadísticos más elaborados sobre matplotlib (barras agrupadas, etc.)
import seaborn as sns

# scipy.stats: para la prueba de hipótesis (t-test) más adelante
from scipy import stats

# Configuración visual: tamaño de fuente y estilo de fondo de los gráficos
sns.set_theme(style="whitegrid")  # equivalente a theme_bw() + theme_set() en ggplot2
plt.rcParams["figure.figsize"] = (12, 6)  # tamaño por defecto de las figuras


## 1. Cargar los datasets

In [ ]:
# Ruta donde están guardados los archivos CSV; edítala según tu entorno
file_path = ""  # ejemplo: "data/" si los CSV están en una subcarpeta llamada data

# Leemos el archivo de transacciones y lo cargamos como DataFrame de pandas
transaction_data = pd.read_csv(file_path + "QVI_transaction_data.csv")

# Leemos el archivo de comportamiento de compra / segmentación de clientes
customer_data = pd.read_csv(file_path + "QVI_purchase_behaviour.csv")


## 2. Exploración inicial de `transaction_data`

In [ ]:
# .info() muestra tipos de dato por columna, cantidad de nulos y uso de memoria
# (equivalente a str() en R): nos deja ver si DATE está como número en vez de fecha
transaction_data.info()


In [ ]:
# .head() muestra las primeras 5 filas para inspeccionar visualmente el contenido
transaction_data.head()


In [ ]:
# .describe() calcula estadísticas descriptivas (media, min, max, percentiles)
# de las columnas numéricas; ayuda a detectar outliers desde ya
transaction_data.describe()


## 3. Convertir la columna `DATE` a formato fecha

La columna `DATE` viene como un número entero (formato serial de Excel/CSV),
donde el día 0 corresponde al 30 de diciembre de 1899. Hay que convertirla a
un tipo de dato `datetime` para poder graficarla y agruparla correctamente.


In [ ]:
# unit="D" indica que el número representa una cantidad de días
# origin="1899-12-30" es el punto de partida que usan Excel y CSV para fechas seriales
transaction_data["DATE"] = pd.to_datetime(
    transaction_data["DATE"], unit="D", origin="1899-12-30"
)

# Verificamos que la conversión fue exitosa mostrando el nuevo tipo de dato
transaction_data["DATE"].head()


## 4. Examinar `PROD_NAME`

Queremos confirmar que todos los productos son efectivamente "papas fritas" (chips)
y no otra categoría que se haya colado en el dataset (por ejemplo, salsas).


In [ ]:
# .value_counts() cuenta cuántas veces aparece cada nombre de producto distinto
# .head(20) limita la salida a los 20 productos más frecuentes, para no saturar la vista
transaction_data["PROD_NAME"].value_counts().head(20)


In [ ]:
# Unimos todos los nombres de producto ÚNICOS en un solo string separado por espacios
# .unique() evita contar la misma palabra muchas veces solo porque el producto se repite
all_names = " ".join(transaction_data["PROD_NAME"].unique())

# .split() separa el string gigante en una lista de palabras individuales (por espacio)
words = all_names.split()

# Mostramos cuántas palabras en total se extrajeron, solo para verificar
len(words)


In [ ]:
# re.sub(r"[^A-Za-z]", "", w) elimina cualquier caracter que NO sea una letra
# (quita dígitos como "175g" -> "g", símbolos como "&", etc.)
clean_words = [re.sub(r"[^A-Za-z]", "", w) for w in words]

# Filtramos los strings que quedaron vacíos después de la limpieza (por ejemplo "175g" -> "")
clean_words = [w for w in clean_words if w != ""]

# Counter cuenta cuántas veces aparece cada palabra limpia y .most_common() las ordena
# de mayor a menor frecuencia
word_freq = Counter(clean_words).most_common()

# Convertimos el resultado a un DataFrame para visualizarlo como tabla ordenada
word_freq_df = pd.DataFrame(word_freq, columns=["word", "count"])

# Mostramos las 20 palabras más frecuentes: aquí es donde detectaríamos "Salsa"
# si hubiera productos que no son chips
word_freq_df.head(20)


## 5. Eliminar productos de salsa (no pertenecen a la categoría chips)

In [ ]:
# .str.lower() convierte el nombre del producto a minúsculas para comparar sin
# importar mayúsculas/minúsculas
# .str.contains("salsa") devuelve True/False si la palabra "salsa" aparece en el nombre
is_salsa = transaction_data["PROD_NAME"].str.lower().str.contains("salsa")

# El operador ~ invierte la máscara booleana (True se vuelve False y viceversa)
# Así nos quedamos solo con las filas donde is_salsa es False (no son salsa)
transaction_data = transaction_data[~is_salsa]

# Confirmamos cuántas filas quedaron después de filtrar
transaction_data.shape


## 6. Resumen estadístico y detección de outliers

Revisamos nulos y valores extremos en las columnas clave antes de seguir.


In [ ]:
# .isna() marca True donde hay valores nulos; .sum() cuenta cuántos hay por columna
# Si todos los valores son 0, significa que no hay datos faltantes
transaction_data.isna().sum()


In [ ]:
# Volvemos a correr describe() ya con los datos filtrados, para ver si PROD_QTY
# (cantidad de productos por transacción) tiene un máximo sospechosamente alto
transaction_data.describe()


In [ ]:
# Filtramos las transacciones donde se compraron exactamente 200 unidades,
# el valor atípico (outlier) que detectamos en el describe() anterior
outlier_transactions = transaction_data[transaction_data["PROD_QTY"] == 200]

# Mostramos esas transacciones para inspeccionarlas manualmente
outlier_transactions


In [ ]:
# Extraemos el/los número(s) de tarjeta de lealtad (LYLTY_CARD_NBR) involucrados
# en la transacción atípica, para revisar el historial completo de ese cliente
loyalty_card_outlier = outlier_transactions["LYLTY_CARD_NBR"].unique()

# .isin() filtra todas las filas donde el número de tarjeta esté en la lista anterior
# Esto nos muestra TODAS las compras hechas por ese cliente en el período analizado
transaction_data[transaction_data["LYLTY_CARD_NBR"].isin(loyalty_card_outlier)]


In [ ]:
# Como el cliente solo tiene esas dos transacciones extremas en todo el año,
# probablemente no es un consumidor minorista normal (podría ser compra comercial)
# Lo excluimos del análisis usando ~ (negación) sobre isin()
transaction_data = transaction_data[
    ~transaction_data["LYLTY_CARD_NBR"].isin(loyalty_card_outlier)
]

# Volvemos a describir los datos para confirmar que el outlier ya no aparece
transaction_data.describe()


## 7. Conteo de transacciones por fecha (buscar días faltantes)

El dataset debería cubrir 365 días (1 jul 2018 - 30 jun 2019). Si el conteo de
fechas únicas da menos de 365, significa que falta al menos un día de datos.


In [ ]:
# .groupby("DATE") agrupa las filas por fecha
# .size() cuenta cuántas transacciones (filas) hay en cada fecha
# .reset_index(name="N") convierte el resultado agrupado de nuevo en un DataFrame,
# nombrando la columna de conteo como "N"
tx_by_day = transaction_data.groupby("DATE").size().reset_index(name="N")

# len() nos dice cuántas fechas distintas hay en total; si es 364 en vez de 365,
# confirma que falta un día
len(tx_by_day)


In [ ]:
# Creamos un rango de fechas completo, día por día, cubriendo TODO el período
# esperado (1 jul 2018 a 30 jun 2019), sin importar si hay datos ese día o no
full_dates = pd.DataFrame({
    "DATE": pd.date_range(start="2018-07-01", end="2019-06-30", freq="D")
})

# Unimos (left join) el rango completo de fechas con el conteo real de transacciones
# how="left" asegura que se mantengan TODAS las fechas del rango, aunque no tengan datos
tx_by_day = full_dates.merge(tx_by_day, on="DATE", how="left")

# Los días sin transacciones quedan como NaN después del merge; los reemplazamos por 0
tx_by_day["N"] = tx_by_day["N"].fillna(0)

# Mostramos las primeras filas para confirmar que ahora sí hay 365 fechas
tx_by_day.shape


In [ ]:
# Creamos la figura y los ejes con un tamaño específico (ancho=12, alto=5 pulgadas)
fig, ax = plt.subplots(figsize=(12, 5))

# Graficamos una línea con el eje X = fecha y el eje Y = número de transacciones
ax.plot(tx_by_day["DATE"], tx_by_day["N"])

# Etiquetas de los ejes, en español, para que el gráfico sea autoexplicativo
ax.set_xlabel("Día")
ax.set_ylabel("Número de transacciones")
ax.set_title("Transacciones a lo largo del tiempo")

# Rotamos las etiquetas del eje X 90 grados para que no se superpongan
plt.xticks(rotation=90)

# Ajusta automáticamente los márgenes para que no se corten las etiquetas
plt.tight_layout()

# Muestra el gráfico en el notebook
plt.show()


In [ ]:
# Filtramos tx_by_day para quedarnos solo con el mes de diciembre de 2018,
# donde vimos un pico y una caída en el gráfico anterior
december = tx_by_day[
    (tx_by_day["DATE"] >= "2018-12-01") & (tx_by_day["DATE"] <= "2018-12-31")
]

# Creamos el gráfico de línea con marcadores en cada punto (marker="o") para ver
# claramente cada día individual de diciembre
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(december["DATE"], december["N"], marker="o")

ax.set_xlabel("Día de diciembre")
ax.set_ylabel("Número de transacciones")
ax.set_title("Transacciones en diciembre 2018 (zoom)")

plt.xticks(rotation=90)
plt.tight_layout()
plt.show()

# Se espera ver un aumento previo a Navidad y una caída a 0 el 25 de diciembre
# (día en que las tiendas están cerradas)


## 8. Crear la variable `PACK_SIZE`

Extraemos el tamaño del empaque (en gramos) directamente del texto en `PROD_NAME`,
ya que el número siempre aparece junto a la "g" de gramos (ej. "175g").


In [ ]:
# .str.extract(r"(\d+)") busca el primer grupo de uno o más dígitos consecutivos
# en cada nombre de producto y lo devuelve como texto
# .astype(int) convierte ese texto extraído en un número entero
transaction_data["PACK_SIZE"] = (
    transaction_data["PROD_NAME"].str.extract(r"(\d+)").astype(int)
)

# .value_counts() cuenta cuántas transacciones hay por cada tamaño de empaque
# .sort_index() ordena el resultado de menor a mayor tamaño (no por frecuencia)
transaction_data["PACK_SIZE"].value_counts().sort_index()


In [ ]:
# Histograma: cada barra representa un rango (bin) de tamaños de empaque
# y su altura indica cuántas transacciones caen en ese rango
transaction_data["PACK_SIZE"].hist(bins=20)

plt.xlabel("Tamaño de empaque (gramos)")
plt.ylabel("Número de transacciones")
plt.title("Distribución de transacciones por tamaño de empaque")
plt.show()


## 9. Crear la variable `BRAND`

La marca suele ser la primera palabra del nombre del producto.


In [ ]:
# .str.split() separa el nombre del producto en una lista de palabras
# .str[0] toma la primera palabra de cada lista (asumimos que ahí está la marca)
# .str.upper() estandariza todo a mayúsculas para evitar duplicados por casing
transaction_data["BRAND"] = (
    transaction_data["PROD_NAME"].str.split().str[0].str.upper()
)

# Revisamos las marcas resultantes y cuántas transacciones tiene cada una
transaction_data["BRAND"].value_counts()


In [ ]:
# Diccionario de corrección: distintas abreviaciones/errores de tipeo que en
# realidad corresponden a la misma marca real (se identifican revisando la lista anterior)
brand_fix = {
    "RED": "RRD",          # "RED" es una abreviación de Red Rock Deli (RRD)
    "SNBTS": "SUNBITES",   # abreviación de Sunbites
    "INFZNS": "INFUZIONS", # abreviación de Infuzions
    "WW": "WOOLWORTHS",    # abreviación de Woolworths
    "SMITH": "SMITHS",     # variación singular/plural de Smiths
    "NCC": "NATURAL",      # abreviación de Natural Chip Co
    "DORITO": "DORITOS",   # variación singular/plural de Doritos
    "GRAIN": "GRNWVES",    # abreviación de Grain Waves
}

# .replace() sustituye cada marca mal escrita por su versión estandarizada,
# usando el diccionario anterior como mapa de reemplazo
transaction_data["BRAND"] = transaction_data["BRAND"].replace(brand_fix)

# Verificamos que las marcas quedaron consolidadas correctamente
transaction_data["BRAND"].value_counts()


## 10. Explorar `customer_data`

In [ ]:
# .info() para ver tipos de dato y nulos en el dataset de clientes
customer_data.info()


In [ ]:
# .describe(include="all") incluye también columnas de texto/categóricas,
# mostrando frecuencias de las categorías más comunes (LIFESTAGE, PREMIUM_CUSTOMER)
customer_data.describe(include="all")


## 11. Unir (merge) transacciones con datos de cliente

In [ ]:
# merge() combina ambos DataFrames usando la columna en común "LYLTY_CARD_NBR"
# how="left" mantiene TODAS las filas de transaction_data, y les agrega la
# información de customer_data cuando encuentra una coincidencia
data = transaction_data.merge(customer_data, on="LYLTY_CARD_NBR", how="left")

# assert lanza un error si la condición es falsa: confirmamos que el merge NO
# generó filas adicionales (lo cual indicaría duplicados en customer_data)
assert len(data) == len(transaction_data), "El merge generó duplicados inesperados"

# Si no hay error, mostramos las dimensiones finales del dataset combinado
data.shape


In [ ]:
# Revisamos si alguna transacción quedó sin datos de cliente después del merge
# (es decir, si hay algún LIFESTAGE o PREMIUM_CUSTOMER nulo)
data[["LIFESTAGE", "PREMIUM_CUSTOMER"]].isna().sum()

# Si el resultado es 0 en ambas columnas, todos los clientes fueron encontrados


In [ ]:
# Guardamos el dataset limpio y combinado en un nuevo archivo CSV,
# útil como punto de partida para el Task 2 del internship
data.to_csv(file_path + "QVI_data.csv", index=False)


## 12. Análisis por segmento de cliente (`LIFESTAGE` x `PREMIUM_CUSTOMER`)

Ahora calculamos las métricas clave para entender qué segmentos impulsan más
las ventas de chips.


In [ ]:
# Agrupamos por las dos dimensiones de segmentación y sumamos TOT_SALES
# (ventas totales) dentro de cada combinación de grupo
sales_by_segment = (
    data.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])["TOT_SALES"]
    .sum()                          # suma las ventas dentro de cada grupo
    .reset_index()                  # convierte el resultado agrupado en DataFrame plano
)

# Gráfico de barras agrupadas: eje X = etapa de vida, color = tipo de comprador,
# altura de barra = ventas totales
plt.figure(figsize=(12, 6))
sns.barplot(data=sales_by_segment, x="LIFESTAGE", y="TOT_SALES", hue="PREMIUM_CUSTOMER")

plt.xticks(rotation=45, ha="right")   # rota etiquetas para que no se sobrepongan
plt.title("Ventas totales por segmento de cliente")
plt.tight_layout()
plt.show()


In [ ]:
# nunique() cuenta cuántas tarjetas de lealtad DISTINTAS hay en cada grupo,
# es decir, cuántos clientes únicos compran en cada segmento
customers_by_segment = (
    data.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])["LYLTY_CARD_NBR"]
    .nunique()
    .reset_index(name="N_CUSTOMERS")
)

plt.figure(figsize=(12, 6))
sns.barplot(data=customers_by_segment, x="LIFESTAGE", y="N_CUSTOMERS", hue="PREMIUM_CUSTOMER")
plt.xticks(rotation=45, ha="right")
plt.title("Número de clientes únicos por segmento")
plt.tight_layout()
plt.show()


In [ ]:
# Función auxiliar: para cada grupo, suma la cantidad total de productos comprados
# (PROD_QTY) y la divide entre el número de clientes únicos del grupo
def avg_units_per_customer(group):
    total_units = group["PROD_QTY"].sum()               # unidades totales del grupo
    unique_customers = group["LYLTY_CARD_NBR"].nunique() # clientes únicos del grupo
    return total_units / unique_customers                # promedio de unidades por cliente

# .apply() ejecuta la función anterior sobre cada grupo definido por groupby()
units_per_customer = (
    data.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])
    .apply(avg_units_per_customer)
    .reset_index(name="AVG_UNITS_PER_CUSTOMER")
)

plt.figure(figsize=(12, 6))
sns.barplot(data=units_per_customer, x="LIFESTAGE", y="AVG_UNITS_PER_CUSTOMER", hue="PREMIUM_CUSTOMER")
plt.xticks(rotation=45, ha="right")
plt.title("Promedio de unidades compradas por cliente, por segmento")
plt.tight_layout()
plt.show()


In [ ]:
# Calculamos el precio pagado por unidad en cada transacción:
# ventas totales de la transacción / cantidad de unidades compradas
data["PRICE_PER_UNIT"] = data["TOT_SALES"] / data["PROD_QTY"]

# Promediamos ese precio por unidad dentro de cada segmento de cliente
avg_price = (
    data.groupby(["LIFESTAGE", "PREMIUM_CUSTOMER"])["PRICE_PER_UNIT"]
    .mean()
    .reset_index()
)

plt.figure(figsize=(12, 6))
sns.barplot(data=avg_price, x="LIFESTAGE", y="PRICE_PER_UNIT", hue="PREMIUM_CUSTOMER")
plt.xticks(rotation=45, ha="right")
plt.title("Precio promedio por unidad, por segmento")
plt.tight_layout()
plt.show()


## 13. Prueba de hipótesis (t-test)

Comparamos el precio por unidad pagado por clientes **Mainstream** (jóvenes y
de mediana edad, solteros/parejas) contra el pagado por clientes **Budget** y
**Premium** del mismo grupo etario, para confirmar si la diferencia observada
en el gráfico anterior es estadísticamente significativa.


In [ ]:
# Filtramos las filas del grupo "Mainstream" dentro de las etapas de vida
# jóvenes/mediana edad solteros o en pareja
mainstream_mask = (
    (data["PREMIUM_CUSTOMER"] == "Mainstream") &
    (data["LIFESTAGE"].isin(["YOUNG SINGLES/COUPLES", "MIDAGE SINGLES/COUPLES"]))
)
mainstream_prices = data.loc[mainstream_mask, "PRICE_PER_UNIT"]

# Filtramos las filas de "Budget" o "Premium" dentro de las mismas etapas de vida,
# para comparar contra el grupo Mainstream
other_mask = (
    (data["PREMIUM_CUSTOMER"].isin(["Budget", "Premium"])) &
    (data["LIFESTAGE"].isin(["YOUNG SINGLES/COUPLES", "MIDAGE SINGLES/COUPLES"]))
)
other_prices = data.loc[other_mask, "PRICE_PER_UNIT"]

# ttest_ind realiza una prueba t de dos muestras independientes
# equal_var=False usa la corrección de Welch (no asume varianzas iguales entre grupos)
t_stat, p_value = stats.ttest_ind(mainstream_prices, other_prices, equal_var=False)

# Mostramos el estadístico t y el p-value resultante
print(f"Estadístico t: {t_stat:.4f}")
print(f"p-value: {p_value:.6f}")

# Si p_value < 0.05, la diferencia de precios es estadísticamente significativa
if p_value < 0.05:
    print("La diferencia de precio por unidad SÍ es estadísticamente significativa.")
else:
    print("La diferencia de precio por unidad NO es estadísticamente significativa.")


## 14. Deep dive: afinidad de marca y tamaño de empaque

Analizamos en detalle el segmento **Mainstream - Young Singles/Couples**
(uno de los que más contribuye a las ventas) para ver si prefiere marcas o
tamaños de empaque particulares, comparado con el resto de la población.


In [ ]:
# Filtramos el segmento objetivo: Mainstream + jóvenes solteros/parejas
target_mask = (
    (data["LIFESTAGE"] == "YOUNG SINGLES/COUPLES") &
    (data["PREMIUM_CUSTOMER"] == "Mainstream")
)
target = data[target_mask]

# El resto de la población es todo lo que NO está en el segmento objetivo
rest = data[~target_mask]

# .value_counts(normalize=True) da la PROPORCIÓN (no el conteo) de cada marca
# dentro del segmento objetivo, para poder comparar en igualdad de condiciones
target_brand_share = target["BRAND"].value_counts(normalize=True)

# Lo mismo pero calculado sobre el resto de la población
rest_brand_share = rest["BRAND"].value_counts(normalize=True)

# El índice de afinidad es la razón entre ambas proporciones:
# > 1 significa que la marca está sobre-representada en el segmento objetivo
# < 1 significa que está sub-representada
affinity_brand = (target_brand_share / rest_brand_share).sort_values(ascending=False)

# Mostramos las 10 marcas con mayor afinidad hacia el segmento objetivo
affinity_brand.head(10)


In [ ]:
# Repetimos el mismo cálculo de afinidad, pero ahora para PACK_SIZE en vez de BRAND
target_pack_share = target["PACK_SIZE"].value_counts(normalize=True)
rest_pack_share = rest["PACK_SIZE"].value_counts(normalize=True)

affinity_pack = (target_pack_share / rest_pack_share).sort_values(ascending=False)

# Tamaños de empaque con mayor afinidad hacia el segmento objetivo
affinity_pack.head(10)


## 15. Conclusiones (a completar según tus resultados)

- **Marca preferida por el segmento objetivo:** revisa `affinity_brand` para
  identificar qué marca(s) tienen el índice de afinidad más alto.
- **Tamaño de empaque preferido:** revisa `affinity_pack` de la misma forma.
- **Diferencia de precio:** usa el resultado del t-test para confirmar si
  el segmento Mainstream paga significativamente más o menos por unidad.

Con esto queda completo el EDA equivalente al template en R, listo para pasar
al Task 2 (recomendaciones de estrategia comercial).
